# 03 Gold (A): analysis + tables

**Owner:** Person A
**Builds:** `gold.dim_customers`, `gold.fact_orders`, `gold.fact_order_items`, `gold.mart_customer_rfm`

**Workflow per question:** question -> explore on silver (read-only) -> finding and decision -> build the gold table -> checks.
The analysis decides what each gold table must contain.

**Rules**
- Exploration queries are read-only; ad-hoc joins on silver are fine there.
- Saved tables follow the gold rule: aggregate one-to-many tables to one row per key BEFORE joining.
- Every build cell is safe to re-run (`DROP TABLE IF EXISTS` first).
- Each finding is 2-3 sentences with numbers.

| Block | Question | Gold table |
|---|---|---|
| A1 | Who are the customers? | `dim_customers` |
| A2 | How does delivery perform? | `fact_orders` |
| A3 | How do customers pay? | payment columns in `fact_orders` |
| A4 | How do revenue and orders change over time? | `fact_order_items` |
| A5 | Who are the best and worst customers? (RFM) | `mart_customer_rfm` |

## A0. Setup

In [4]:
# Importing libraries
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# db.py lives in notebooks/, two levels above this notebook's folder
# (adjust .parent chain if this notebook is placed in a different folder)
NOTEBOOKS_DIR = Path.cwd().parent.parent
sys.path.append(str(NOTEBOOKS_DIR))

from db import get_engine
engine = get_engine()


In [5]:
def q(sql):
    """Run a SELECT and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql):
    """Run CREATE/DROP/INSERT and commit."""
    with engine.begin() as conn:
        conn.execute(text(sql))

---

## A1. gold.dim_customers
- **Explore:** customers vs unique people, repeat-buyer share, customers per state.
- **Base:** `silver.customers`; **Joins:** none.
- **Columns:** `customer_id`, `customer_unique_id`, `customer_zip_code_prefix`, `customer_city`, `customer_state`.
- **Note:** a real person is `customer_unique_id` (one person can have several `customer_id` values). Use `DISTINCTCOUNT(customer_unique_id)` in Power BI for customer counts.

In [8]:
# Rebuild gold.dim_customers (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_customers"))
    conn.execute(text("""
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix,
            customer_city,
            customer_state
        INTO gold.dim_customers
        FROM silver.customers
    """))

---

## A2. gold.fact_orders
- **Explore:** share of delivered orders, delivery days, late share by state.
**`gold.fact_orders` spec**
- **Columns:** `order_id`, `customer_id`, `order_status`, `order_date` (DATE), `approved_date`, `delivered_carrier_date`, `delivered_customer_date`, `estimated_delivery_date`, `delivery_days`, `delay_days`, `is_late` (BIT), `total_payment`, `payment_installments_max`, `main_payment_type`.
- **Logic:** `delivery_days` = delivered date minus purchase date; `delay_days` = delivered date minus estimated date; `is_late` = `delay_days > 0`. Delivery columns stay `NULL` for undelivered orders (`is_late` is NULL, not 0).

In [15]:
# Step 1: one row per order with total payment and max installments
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_pay"))
    conn.execute(text("""
        SELECT order_id,
               SUM(payment_value)        AS total_payment,
               MAX(payment_installments) AS payment_installments_max
        INTO gold._tmp_pay
        FROM silver.order_payments
        GROUP BY order_id
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_pay")

,rows_,orders
0,99440,99440


In [16]:
# Step 2: main payment type = type with the highest value in the order
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_main_type"))
    conn.execute(text("""
        WITH t AS (
            SELECT order_id, payment_type,
                   ROW_NUMBER() OVER (
                       PARTITION BY order_id
                       ORDER BY SUM(payment_value) DESC, payment_type
                   ) AS rn
            FROM silver.order_payments
            GROUP BY order_id, payment_type
        )
        SELECT order_id, payment_type AS main_payment_type
        INTO gold._tmp_main_type
        FROM t
        WHERE rn = 1
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_main_type")

,rows_,orders
0,99440,99440


In [17]:
# Step 3: orders with delivery_days and delay_days (NULL if not delivered)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_base"))
    conn.execute(text("""
        SELECT order_id,
               customer_id,
               order_status,
               CAST(order_purchase_timestamp AS DATE) AS order_date,
               order_approved_at                      AS approved_date,
               order_delivered_carrier_date           AS delivered_carrier_date,
               order_delivered_customer_date          AS delivered_customer_date,
               order_estimated_delivery_date          AS estimated_delivery_date,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_purchase_timestamp AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delivery_days,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                         AND order_estimated_delivery_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_estimated_delivery_date AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delay_days
        INTO gold._tmp_base
        FROM silver.orders
    """))

q("SELECT COUNT(*) AS rows_ FROM gold._tmp_base")

,rows_
0,99441


In [18]:
# Step 4: final table = base + payment total + main payment type
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_orders"))
    conn.execute(text("""
        SELECT b.order_id,
               b.customer_id,
               b.order_status,
               b.order_date,
               b.approved_date,
               b.delivered_carrier_date,
               b.delivered_customer_date,
               b.estimated_delivery_date,
               b.delivery_days,
               b.delay_days,
               -- NULL (not 0) when delay_days is unknown
               CAST(CASE WHEN b.delay_days IS NULL THEN NULL
                         WHEN b.delay_days > 0 THEN 1
                         ELSE 0 END AS BIT)  AS is_late,
               p.total_payment,
               p.payment_installments_max,
               m.main_payment_type
        INTO gold.fact_orders
        FROM gold._tmp_base b
        LEFT JOIN gold._tmp_pay p       ON p.order_id = b.order_id
        LEFT JOIN gold._tmp_main_type m ON m.order_id = b.order_id
    """))

q("""
SELECT (SELECT COUNT(*) FROM silver.orders)    AS silver_rows,
       (SELECT COUNT(*) FROM gold.fact_orders) AS gold_rows
""")

,silver_rows,gold_rows
0,99441,99441


In [19]:
# Step 5: drop the staging tables
with engine.begin() as conn:
    for t in ["_tmp_pay", "_tmp_main_type", "_tmp_base"]:
        conn.execute(text(f"DROP TABLE IF EXISTS gold.{t}"))

---

## A4. gold.fact_order_items
- **Explore:** monthly revenue and orders on silver, revenue by order status, orphan check.
- **Joins:** `LEFT JOIN silver.orders` on `order_id` (one row per order, so no duplication) for `customer_id`, `order_date`, `order_status`.
- **Columns:** `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `order_date`, `order_status`, `price`, `freight_value`, `item_total` (price + freight).


In [27]:
# Build gold.fact_order_items (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_order_items"))
    conn.execute(text("""
        SELECT i.order_id,
               i.order_item_id,
               i.product_id,
               i.seller_id,
               o.customer_id,
               CAST(o.order_purchase_timestamp AS DATE) AS order_date,
               o.order_status,
               i.price,
               i.freight_value,
               i.price + i.freight_value                AS item_total
        INTO gold.fact_order_items
        FROM silver.order_items i
        LEFT JOIN silver.orders o ON o.order_id = i.order_id
    """))

---

## A5. gold.mart_customer_rfm (RFM)
- **Joins:** `gold.fact_orders` + `gold.dim_customers` (to get `customer_unique_id`) + `gold.fact_order_items` aggregated to one row per `order_id` first.
- **Columns:** `customer_unique_id`, `recency_days`, `frequency`, `monetary`, `r_score` (1-5), `f_score` (1-3), `m_score` (1-5), `segment`.
- **Logic:** reference date = last delivered order + 1 day. R and M use `NTILE(5)`. F: 1 order = 1, 2 orders = 2, 3+ = 3.
- **Segments:** Champions (f >= 2 and r >= 4), Loyal (f >= 2), New high value (r >= 4 and m >= 4), New (r >= 4), Lost (r = 1), Needs attention (rest).

In [ ]:
# Build gold.mart_customer_rfm in one query (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_customer_rfm"))
    conn.execute(text("""
        WITH items AS (
            -- Aggregate items to one row per order BEFORE joining
            SELECT order_id, SUM(price) AS order_value
            FROM gold.fact_order_items
            GROUP BY order_id
        ),
        ord AS (
            SELECT c.customer_unique_id, f.order_id, f.order_date,
                   COALESCE(i.order_value, 0) AS order_value
            FROM gold.fact_orders f
            JOIN gold.dim_customers c ON c.customer_id = f.customer_id
            LEFT JOIN items i         ON i.order_id = f.order_id
            WHERE f.order_status = 'delivered'
        ),
        base AS (
            -- Reference date = day after the last order in the data
            SELECT customer_unique_id,
                   DATEDIFF(day, MAX(order_date), (SELECT DATEADD(day, 1, MAX(order_date)) FROM ord)) AS recency_days,
                   COUNT(DISTINCT order_id) AS frequency,
                   SUM(order_value)         AS monetary
            FROM ord
            GROUP BY customer_unique_id
        ),
        s AS (
            SELECT *,
                   NTILE(5) OVER (ORDER BY recency_days DESC, customer_unique_id) AS r_score,
                   CASE WHEN frequency >= 3 THEN 3 WHEN frequency = 2 THEN 2 ELSE 1 END AS f_score,
                   NTILE(5) OVER (ORDER BY monetary ASC, customer_unique_id)      AS m_score
            FROM base
        )
        SELECT customer_unique_id, recency_days, frequency, monetary,
               r_score, f_score, m_score,
               CASE WHEN f_score >= 2 AND r_score >= 4 THEN 'Champions'
                    WHEN f_score >= 2                  THEN 'Loyal'
                    WHEN r_score >= 4 AND m_score >= 4 THEN 'New high value'
                    WHEN r_score >= 4                  THEN 'New'
                    WHEN r_score = 1                   THEN 'Lost'
                    ELSE 'Needs attention' END         AS segment
        INTO gold.mart_customer_rfm
        FROM s
    """))